In [ ]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

## 1. Imports and Model Loading

In [ ]:
import sys
import os

parent_dir = os.path.dirname(os.getcwd()) 
sys.path.insert(0, parent_dir)

from utils import (
    setup_sam_3d_body, setup_visualizer, 
    visualize_2d_results, visualize_3d_mesh, save_mesh_results, 
    display_results_grid, process_image_with_mask
)

# Set up SAM 3D Body estimator
estimator = setup_sam_3d_body(hf_repo_id="facebook/sam-3d-body-dinov3")
# Set up visualizer
visualizer = setup_visualizer()

## 2. Process Image and Get Outputs

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Load and process the image
image_path = "images/diego.jpeg"  # Relative to notebook folder
img_cv2 = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB)

# Process the image with SAM 3D Body
print("Processing image with SAM 3D Body...")
outputs = estimator.process_one_image(image_path)

print(f"Number of people detected: {len(outputs)}")
print(f"Output keys for first person: {list(outputs[0].keys()) if outputs else 'No people detected'}")

# Display the original image
plt.figure(figsize=(10, 6))
plt.imshow(img_rgb)
plt.axis('off')
plt.title('Original Image')
plt.show()

## 3. 2D Visualization - Keypoints and Bounding Boxes

In [ ]:
# Visualize 2D results using utils
if outputs:
    vis_results = visualize_2d_results(img_cv2, outputs, visualizer)
    
    # Display results using grid function
    titles = [f'Person {i} - 2D Keypoints & BBox' for i in range(len(vis_results))]
    display_results_grid(vis_results, titles, figsize_per_image=(6, 6))
else:
    print("No people detected in the image")

## 4. 3D Mesh Visualization - Overlay and Side View

In [ ]:
if outputs:
    mesh_results = visualize_3d_mesh(img_cv2, outputs, estimator.faces)
    
    # Display results
    for i, combined_img in enumerate(mesh_results):
        combined_rgb = cv2.cvtColor(combined_img, cv2.COLOR_BGR2RGB)
        
        plt.figure(figsize=(20, 5))
        plt.imshow(combined_rgb)
        plt.title(f'Person {i}: Original | Mesh Overlay | Front View | Side View')
        plt.axis('off')
        plt.show()
else:
    print("No people detected for 3D mesh visualization")

## 5. Save 3D Mesh Files and Results

In [ ]:
if outputs:
    # Get image name without extension
    image_name = os.path.splitext(os.path.basename(image_path))[0]
    
    # Create output directory
    output_dir = f"output/{image_name}"

    # Save all results (PLY meshes, overlay images, bbox images)
    ply_files = save_mesh_results(img_cv2, outputs, estimator.faces, output_dir, image_name)
    
    print(f"\n=== Saved Results for {image_name} ===")
    print(f"Output directory: {output_dir}")
    print(f"Number of PLY files created: {len(ply_files)}")
    
else:
    print("No results to save - no people detected")

# video mesh

In [ ]:
import cv2
import numpy as np
import warnings
import sys
import os
from tqdm import tqdm

warnings.filterwarnings('ignore')

class SuppressPrints:
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout.close()
        sys.stdout = self._original_stdout

# Extract frames
video_path = "images/diego.mp4"
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

frames = []
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    if frame_idx % 5 == 0:
        frames.append(frame)
    frame_idx += 1

cap.release()
print(f"Extracted {len(frames)} frames (expected ~{int(cap.get(cv2.CAP_PROP_FRAME_COUNT) / 5)})")

frames_rgb = [cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in frames]

# Process in batches
batch_size = 8
all_outputs = []

for i in tqdm(range(0, len(frames_rgb), batch_size), desc="Processing batches"):
    batch = frames_rgb[i:i+batch_size]
    
    for frame_rgb in batch:
        with SuppressPrints():
            outputs = estimator.process_one_image(frame_rgb)
        all_outputs.append(outputs if outputs else [])

# Tracking
tracked_outputs = []
prev_pos = None
max_dist = 150

for outputs in tqdm(all_outputs, desc="Tracking"):
    if outputs:
        if prev_pos is None:
            selected = outputs[0]
        else:
            selected = min(outputs, key=lambda p: np.linalg.norm(
                p['pred_vertices'].mean(axis=0)[:2] - prev_pos))
            if np.linalg.norm(selected['pred_vertices'].mean(axis=0)[:2] - prev_pos) > max_dist:
                tracked_outputs.append([])
                continue
        
        prev_pos = selected['pred_vertices'].mean(axis=0)[:2]
        tracked_outputs.append([selected])
    else:
        tracked_outputs.append([])

print(f"Tracked: {sum(1 for o in tracked_outputs if o)}/{len(frames)}")
all_outputs = tracked_outputs

In [ ]:
def create_mesh_video(frames, all_outputs, estimator, fps, output_path=None, frame_skip=5):
    """
    Create video with mesh overlay from processed frames.
    
    Args:
        frames: List of BGR frames
        all_outputs: List of SAM 3D Body outputs per frame
        estimator: SAM 3D Body estimator (for faces)
        fps: Original video FPS
        output_path: Path for output video (default: auto-generated)
        frame_skip: Frame skip used during extraction (default: 5)
    
    Returns:
        Path to created video
    """
    import subprocess
    import os
    import shutil
    from tqdm import tqdm
    import cv2
    
    # Set default output path
    if output_path is None:
        output_path = os.path.expanduser("~/SAM-mesh/sam-3d-body/notebook/output/mesh_video.mp4")
    output_path = os.path.expanduser(output_path)
    
    # Setup frame directory
    output_dir = os.path.splitext(output_path)[0] + "_frames"
    
    # Clear old frames
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)
    
    # Save frames
    for i, (frame, outputs) in enumerate(tqdm(zip(frames, all_outputs), 
                                              total=len(frames), 
                                              desc="Saving frames")):
        if outputs:
            mesh_vis = visualize_3d_mesh(frame, outputs, estimator.faces)
            cv2.imwrite(f"{output_dir}/frame_{i:04d}.jpg", mesh_vis[0])
        else:
            cv2.imwrite(f"{output_dir}/frame_{i:04d}.jpg", frame)
    
    # Create video
    print("\nCreating video with ffmpeg...")
    playback_fps = fps / frame_skip
    
    subprocess.run([
        'ffmpeg', '-y', '-loglevel', 'error',
        '-framerate', str(playback_fps),
        '-i', f'{output_dir}/frame_%04d.jpg',
        '-c:v', 'libx264', '-crf', '23', '-pix_fmt', 'yuv420p',
        output_path
    ], check=True)
    
    print(f"Video created: {output_path}")
    return output_path



In [ ]:

video_path = create_mesh_video(
    frames,       # List of BGR frames
    all_outputs,  # List of SAM 3D Body outputs per frame
    estimator,    # SAM 3D Body estimator (for faces)
    fps,          # Original-video FPS
    output_path = "~/SAM-mesh/sam-3d-body/notebook/output/mesh_chorevideo.mp4",  # Path for output video (default: auto-generated)
    frame_skip = 5)   #: Frame skip used during extraction (default: 5)frames, all_outputs, estimator, fps)


# with person selection

In [ ]:
import cv2
import numpy as np
import warnings
import sys
import os
from tqdm import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

class SuppressPrints:
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout.close()
        sys.stdout = self._original_stdout

# Extract frames - FIXED: removed double read
video_path = "images/diego.mp4"
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

frames = []
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    if frame_idx % 5 == 0:
        frames.append(frame)
    frame_idx += 1

cap.release()
print(f"Extracted {len(frames)} frames")

# Show first frame and let user choose
frame_rgb = cv2.cvtColor(frames[0], cv2.COLOR_BGR2RGB)
with SuppressPrints():
    first_outputs = estimator.process_one_image(frame_rgb)

if len(first_outputs) > 1:
    print(f"\n{len(first_outputs)} people detected in first frame")
    vis = visualize_2d_results(frames[0], first_outputs, visualizer)
    
    fig, axes = plt.subplots(1, len(vis), figsize=(6*len(vis), 6))
    if len(vis) == 1:
        axes = [axes]
    for idx, (img, ax) in enumerate(zip(vis, axes)):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f'Person {idx}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    person_idx = int(input(f"Which person to track? (0-{len(first_outputs)-1}): "))
    initial_person = first_outputs[person_idx]
else:
    print("1 person detected - auto-selecting")
    initial_person = first_outputs[0]



In [ ]:
# Convert frames to RGB
frames_rgb = [cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in frames]

# Process in batches
batch_size = 8
all_outputs = []

for i in tqdm(range(0, len(frames_rgb), batch_size), desc="Processing batches"):
    batch = frames_rgb[i:i+batch_size]
    
    for frame_rgb in batch:
        with SuppressPrints():
            outputs = estimator.process_one_image(frame_rgb)
        all_outputs.append(outputs if outputs else [])

# Tracking with initial person
tracked_outputs = []
prev_pos = initial_person['pred_vertices'].mean(axis=0)[:2]
tracked_outputs.append([initial_person])
max_dist = 150

for outputs in tqdm(all_outputs[1:], desc="Tracking"):
    if outputs:
        selected = min(outputs, key=lambda p: np.linalg.norm(
            p['pred_vertices'].mean(axis=0)[:2] - prev_pos))
        
        if np.linalg.norm(selected['pred_vertices'].mean(axis=0)[:2] - prev_pos) < max_dist:
            prev_pos = selected['pred_vertices'].mean(axis=0)[:2]
            tracked_outputs.append([selected])
        else:
            tracked_outputs.append([])
    else:
        tracked_outputs.append([])

print(f"Tracked: {sum(1 for o in tracked_outputs if o)}/{len(frames)}")
all_outputs = tracked_outputs

# Create video
import subprocess

output_dir = os.path.expanduser("~/SAM-mesh/sam-3d-body/notebook/output/mesh_frames")
os.makedirs(output_dir, exist_ok=True)

for i, (frame, outputs) in enumerate(tqdm(zip(frames, all_outputs), total=len(frames), desc="Saving frames")):
    if outputs:
        mesh_vis = visualize_3d_mesh(frame, outputs, estimator.faces)
        cv2.imwrite(f"{output_dir}/frame_{i:04d}.jpg", mesh_vis[0])
    else:
        cv2.imwrite(f"{output_dir}/frame_{i:04d}.jpg", frame)

print("\nCreating video with ffmpeg...")
output_video = os.path.expanduser("~/SAM-mesh/sam-3d-body/notebook/output/mesh_video.mp4")
subprocess.run([
    'ffmpeg', '-y', '-framerate', str(fps/5),
    '-i', f'{output_dir}/frame_%04d.jpg',
    '-c:v', 'libx264', '-crf', '23', '-pix_fmt', 'yuv420p',
    output_video
])
print(f"Video: {output_video}")

In [ ]:
# Save frames as images first
output_dir = os.path.expanduser("~/SAM-mesh/sam-3d-body/notebook/output/mesh_frames")
os.makedirs(output_dir, exist_ok=True)

print("Saving frames...")
for i, (frame, outputs) in enumerate(zip(frames, all_outputs)):
    if outputs:
        mesh_vis = visualize_3d_mesh(frame, outputs, estimator.faces)
        cv2.imwrite(f"{output_dir}/frame_{i:04d}.jpg", mesh_vis[0])
    else:
        cv2.imwrite(f"{output_dir}/frame_{i:04d}.jpg", frame)
    
    if i % 20 == 0:
        print(f"Saved {i}/{len(frames)}")

print("\nCreating video with ffmpeg...")
# Create video with H.264 compression
output_video = os.path.expanduser("~/SAM-mesh/sam-3d-body/notebook/output/mesh_video.mp4")
cmd = [
    'ffmpeg', '-y',
    '-framerate', str(fps/5),
    '-i', f'{output_dir}/frame_%04d.jpg',
    '-c:v', 'libx264',
    '-crf', '23',  # Quality (lower = better, 18-28 typical)
    '-pix_fmt', 'yuv420p',
    output_video
]

subprocess.run(cmd)
print(f"Video created: {output_video}")